# 04. Tools 단독 테스트

Day 2에서 만들 도구 4개를 Agent에 결합하기 전에 **각 도구가 단독으로 잘 동작하는지** 확인.

**진행 순서**
1. `search_guam_guide` (RAG 도구) — 03_rag_test의 retriever를 @tool로 감싼 것
2. `get_guam_weather` (예정)
3. `convert_currency` (예정)
4. `search_web` (예정, 선택)

각 도구는 단독으로 string을 반환해야 함 (옵션 A 패턴: 도구 안에서 LLM 호출 X, Agent가 답변 작성).

## 1. 환경 설정 + import 테스트

`src/guam_chatbot/tools/guide.py`에서 `search_guam_guide`를 가져와서 LangChain Tool 객체로 잘 만들어졌는지 확인.

In [1]:
import sys
from pathlib import Path

# 노트북에서 src/ 모듈을 import할 수 있도록 sys.path에 추가
# 노트북 작업 디렉토리는 practice/, 그 한 단계 위의 src/를 path에 넣음
sys.path.insert(0, str(Path.cwd().parent / "src"))

# guide.py에서 만든 @tool 함수 import
from guam_chatbot.tools.guide import search_guam_guide

# @tool이 함수를 LangChain Tool 객체로 변환했는지 확인
print(f"도구 이름: {search_guam_guide.name}")
print(f"\n도구 설명 (LLM이 도구 선택 시 읽는 문구):")
print(search_guam_guide.description)
print(f"\n도구 입력 스키마: {search_guam_guide.args}")

도구 이름: search_guam_guide

도구 설명 (LLM이 도구 선택 시 읽는 문구):
괌 여행 정보(리조트, 식당, 관광 명소, 액티비티, 교통, 안전 등)를 검색합니다.

    가족 여행자가 물어볼 만한 괌 현지 정보를 답할 때 사용하세요.
    예: 어떤 리조트가 가족에게 좋은지, 추천 식당, 아이들과 가볼 만한 명소,
        공항-시내 이동 방법, 안전 주의사항 등.

    이 도구는 미리 수집한 가이드북 자료(위키백과, Wikivoyage, 나무위키 등)에서
    검색합니다. 실시간 정보(현재 날씨, 환율, 최신 가격)는 다루지 않으므로
    그런 질문에는 다른 도구를 사용하세요.

도구 입력 스키마: {'query': {'title': 'Query', 'type': 'string'}}


## 2. 도구 동작 테스트 — Q1 회귀 검증

03_rag_test에서 검증한 Q1("PIC 리조트 어때?")을 도구 호출 방식으로 다시 돌려서, RAG 도구가 같은 검색 결과를 내는지 확인.

**확인할 것**
- 검색 시간 (도구는 LLM 호출이 없어서 03보다 훨씬 빠를 것 — 2.82초 → 1초 이내 예상)
- 반환값이 `[category] title` 라벨이 붙은 string인지
- top 3에 PIC 청크가 들어오는지 (03 결과와 일치해야 함)

In [2]:
import time

# 03_rag_test 시연 Q1과 같은 질문 — 회귀 검증 효과
question = "PIC 리조트 어때? 가족이 가도 좋아?"

start = time.time()
result = search_guam_guide.invoke(question)
elapsed = time.time() - start

print(f"Q: {question}")
print(f"⏱️  검색 시간: {elapsed:.2f}초\n")
print("=" * 60)
print("도구 반환값 (Agent의 LLM이 받게 될 raw 결과):")
print("=" * 60)
print(result)

Q: PIC 리조트 어때? 가족이 가도 좋아?
⏱️  검색 시간: 0.13초

도구 반환값 (Agent의 LLM이 받게 될 raw 결과):
[리조트] 퍼시픽 아일랜드 클럽 (PIC) - 나무위키
1. 개요

Pacific Islands Club, PIC

괌 및 북마리아나 제도에 위치한 미국의 복합형 리조트. 흔히 PIC라는 약칭으로 불린다. 괌 및 사이판에 여행하는 관광객들이 가장 많이 찾는 숙박시설이다.

한국인, 일본인 관광객들이 가장 많이 숙박하며, 각 리조트에는 한국인, 일본인 직원 및 현지인 중 한국어 및 일본어 구사가 가능한 직원이 배치되어있다.

워낙 한국인 관광객이 많이 찾다 보니, 한국어 사이트도 개설되어 있다.[5]
2. PIC 괌
괌 투몬만에 위치한 퍼시픽 아일랜드 클럽 괌
PIC 하면 생각나는 리조트이며, PIC 리조트 중 가장 큰 리조트이다. 괌에서 최대 규모를 자랑하며, 괌을 찾는 관광객들이 가장 많이 애용한다. 특히 가족 단위 관광객들에게 특화되어 있으며 다양한 레크리에이션과, 괌 최대 규모의 수영장, 그리고 어린이 관광객을 위한 체험 프로그램이 마련되어 있다.

투숙하는 기간 내내 매일 석식뷔페에 간다면 알 수 있겠지만 오늘 봤던 한국인 손님은 다음날엔 없고 전날에 못봤던 다른 한국인 손님으로 채워지는 등 매일매일 회전하듯이 바뀐다.[6] 그만큼 한국인이 많이 찾는단 얘기다.

뷔페는 관광객 기호에 따라 한식, 중식, 일식, 양식, 현지식으로 다양하게 구성되어 있다.

다만 호텔 시설이 오래됐다는 단점이 있다, 다만 호텔의 서비스, 워터파크, 아이들 놀이시설 같이 호텔과 차별되는 이 호텔의 강점으로 커버가 가능하다.
3. PIC 사이판
키즈 동반 및 워터슬라이드를 즐기는 성인이라면, PIC에서 모든 것을 해결할 수 있다. 국내에서 긴 줄을 기다려야 탈 수 있는 대형 워터 슬라이드 3종이 기다림 없이 가능하고 스릴도 대단하다. 유수풀, 소형 슬라이드, 인공 파도 타기(포인트 브레이크) 등도 즐길 수 있다

그 외에 카약, 패들보트,

### guide.py 검증 결과 ✅

| 항목 | 결과 |
|---|---|
| import 동작 | 정상 (sys.path에 src/ 추가 후) |
| @tool 변환 | name·description·args 모두 자동 생성 |
| Q1 검색 | 0.13초, top 3 모두 PIC 청크 |

**알려진 이슈** (Day 2 Agent 단계에서 해결 예정)
- 괌/사이판 PIC 정보 혼재 → Agent 시스템 프롬프트의 페르소나로 가드
- Q4("4박 5일 30만원") RAG 부분은 Agent 통합 시 자연 검증 (Day 1 인계 정책)

**다음 도구**: get_guam_weather (OpenWeatherMap)

## 3. weather 도구 — import + 메타정보 확인

`src/guam_chatbot/tools/weather.py`의 `get_guam_weather`를 가져와서 LangChain Tool 객체로 잘 만들어졌는지 확인.

**확인할 것**
- guide.py와 docstring에서 책임 경계가 명확히 분리되어 있는지 ("실시간 정보 vs 가이드북 정보")
- 도구 이름·설명·입력 스키마가 자동 생성되는지
- (셀 3에서 이미 sys.path를 추가했으므로 여기선 import만)

In [3]:
# weather.py에서 만든 @tool 함수 import
# sys.path는 셀 3에서 이미 추가됨
from guam_chatbot.tools.weather import get_guam_weather

# @tool이 함수를 LangChain Tool 객체로 변환했는지 확인
print(f"도구 이름: {get_guam_weather.name}")
print(f"\n도구 설명 (LLM이 도구 선택 시 읽는 문구):")
print(get_guam_weather.description)
print(f"\n도구 입력 스키마: {get_guam_weather.args}")

도구 이름: get_guam_weather

도구 설명 (LLM이 도구 선택 시 읽는 문구):
괌의 현재 및 5일치 날씨 예보를 가져옵니다.

    사용자가 괌의 날씨, 기온, 강수확률, 우천 여부 등 시점이 있는 실시간 정보를
    물어볼 때 사용하세요.
    예: 5월 둘째 주 괌 날씨 어때?, 내일 비 와?, 오늘 우비 챙겨야 해?

    이 도구는 5일 후까지의 예보만 제공합니다. 그 이후의 날씨나 일반 기후 정보
    (우기·건기, 평균 기온 등)는 search_guam_guide를 사용하세요.

도구 입력 스키마: {'query': {'title': 'Query', 'type': 'string'}}


## 4. weather 도구 동작 테스트 — Q2 시점 정보 검증

03_rag_test의 시연 Q2("5월 둘째 주 괌 날씨")를 도구 호출로 다시 확인. 03에서는 RAG 한계로 시점·수치 환각(예: "70~100mm", "오후/저녁")이 있었는데, 도구를 통해 실제 예보를 받아오는지 검증.

**확인할 것**
- 도구 호출 성공 (⚠️ API 키 발급 직후라면 401 받을 수 있음 — 10분~2시간 활성화 대기 후 재시도)
- 5일치 일별 요약 string이 반환되는지 (오늘 ~ 5일 후 부근)
- 평균/최저/최고 기온, 비 확률, 한국어 description 모두 채워졌는지
- 응답 시간 (네트워크 1회 호출 + JSON 파싱이라 1초 이내 예상)
- "5일 한도" 안내가 끝에 붙는지 (Agent가 5/17 이후 질문에 가이드 도구로 fallback 판단할 근거)

In [4]:
import time

# 03_rag_test 시연 Q2와 같은 질문 — RAG 한계가 드러났던 질문
question = "5월 둘째 주 괌 날씨 어때? 우비 챙겨야 해?"

start = time.time()
result = get_guam_weather.invoke(question)
elapsed = time.time() - start

print(f"Q: {question}")
print(f"⏱️  응답 시간: {elapsed:.2f}초\n")
print("=" * 60)
print("도구 반환값 (Agent의 LLM이 받게 될 raw 결과):")
print("=" * 60)
print(result)

Q: 5월 둘째 주 괌 날씨 어때? 우비 챙겨야 해?
⏱️  응답 시간: 0.81초

도구 반환값 (Agent의 LLM이 받게 될 raw 결과):
괌 (Tumon) 5일 일별 날씨 예보 (괌 현지 시각 기준)
2026-05-08 (금) — 평균 28°C (최저 27 / 최고 28), 비 확률 20%, 온흐림
2026-05-09 (토) — 평균 27°C (최저 25 / 최고 28), 비 확률 100%, 실 비
2026-05-10 (일) — 평균 27°C (최저 26 / 최고 29), 비 확률 100%, 실 비
2026-05-11 (월) — 평균 27°C (최저 26 / 최고 28), 비 확률 100%, 튼구름
2026-05-12 (화) — 평균 27°C (최저 26 / 최고 28), 비 확률 100%, 실 비
2026-05-13 (수) — 평균 27°C (최저 26 / 최고 28), 비 확률 0%, 구름조금

※ 본 예보는 5일 한도이며, 그 이후 날짜는 알 수 없습니다.


### weather.py 검증 결과 ✅

| 항목 | 결과 |
|---|---|
| import 동작 | 정상 (셀 3에서 추가한 sys.path 재사용) |
| @tool 변환 | name·description·args 자동 생성 |
| docstring 3단 구조 | 한 줄 설명 + 사용 예시 + 다른 도구 경계 모두 인식 |
| Q2 응답 시간 | 0.81초 (예상 1초 이내 통과) |
| 일별 요약 string | 5/8 ~ 5/13 6일치 정상 출력, "5일 한도" 안내 포함 |
| API 키 활성 | 401 안 받음 → 정상 활성 |

**알려진 이슈** (Day 2 후반 Agent 통합 단계에서 자연 검증 예정)
- 5일 의도였으나 실제 6일치 출력 — UTC↔현지 변환 + 호출 시각으로 캘린더상 6일에 걸침. 버그 아님, 오히려 정보 풍부. 다만 마지막 날 슬롯 수 적어 통계 부정확 가능 (예: 5/13 비 확률 0%는 슬롯 1~2개에 의존)
- `lang=kr` 한국어 번역 어색 ("실 비" / "온흐림" / "튼구름") — OpenWeatherMap 측 품질 문제. Agent의 LLM이 raw 데이터를 받아 답변 작성 시 자연 처리 가능성 (가설, 검증 필요)

**다음 도구**: convert_currency (한국은행 ECOS)

## 5. currency 도구 — import + 메타정보 확인

`src/guam_chatbot/tools/currency.py`의 `convert_currency`를 가져와서 LangChain Tool 객체로 잘 만들어졌는지 확인.

**확인할 것**
- 도구 이름·설명·입력 스키마가 자동 생성되는지
- docstring에서 "환산 계산은 호출자가" 가이드가 명확한지 (옵션 A 패턴)
- 다른 통화 쌍(JPY, EUR) 미지원이 명시되는지

In [5]:
# currency.py에서 만든 @tool 함수 import
# sys.path는 셀 3에서 이미 추가됨
from guam_chatbot.tools.currency import convert_currency

print(f"도구 이름: {convert_currency.name}")
print(f"\n도구 설명 (LLM이 도구 선택 시 읽는 문구):")
print(convert_currency.description)
print(f"\n도구 입력 스키마: {convert_currency.args}")

도구 이름: convert_currency

도구 설명 (LLM이 도구 선택 시 읽는 문구):
미국 달러(USD)와 한국 원화(KRW) 사이 환율을 가져옵니다.

    사용자가 환율, 미국 달러, 원화 환산을 물어볼 때 사용하세요.
    예: 100달러면 한국 돈 얼마?, 환율 알려줘, 1달러 몇 원?

    이 도구는 한국은행이 고시하는 매매기준율을 반환합니다 (영업일 기준,
    주말·공휴일 데이터 없음). 환율값만 제공하므로 100 × 환율 같은 환산
    계산은 호출자가 직접 수행해야 합니다. 다른 통화 쌍(JPY, EUR 등)은
    지원하지 않습니다.

도구 입력 스키마: {'query': {'title': 'Query', 'type': 'string'}}


## 6. currency 도구 동작 테스트 — Q3 환율 조회

시연 시나리오 #3("100달러면 한국 돈 얼마야?")에서 Agent가 호출할 도구. 옵션 A 패턴이라 도구는 raw 환율만 반환하고, "100 × 환율" 계산은 Agent의 LLM이 직접 수행함. 여기선 도구가 raw 환율을 잘 가져오는지만 검증.

**확인할 것**
- ECOS API 호출 성공 (⚠️ 첫 호출 시 인증키 활성 대기 가능 — 발급 후 1일 이내)
- 최근 5영업일치 USD/KRW 환율 string 반환
- 가장 최근 영업일이 마지막 줄 안내에 명시되는지
- 응답 시간 (네트워크 1회 + JSON 파싱이라 1초 이내 예상)
- 영업일·지연 안내 문구

In [6]:
import time

# 시연 #3과 같은 의도의 질문 — 도구가 raw 환율만 반환하는지 확인
question = "100달러면 한국 돈 얼마야?"

start = time.time()
result = convert_currency.invoke(question)
elapsed = time.time() - start

print(f"Q: {question}")
print(f"⏱️  응답 시간: {elapsed:.2f}초\n")
print("=" * 60)
print("도구 반환값 (Agent의 LLM이 받게 될 raw 결과):")
print("=" * 60)
print(result)

Q: 100달러면 한국 돈 얼마야?
⏱️  응답 시간: 0.55초

도구 반환값 (Agent의 LLM이 받게 될 raw 결과):
USD/KRW 환율 (한국은행 매매기준율 — 영업일 기준)
2026-05-08 (금) — 1 USD = 1450.80 KRW
2026-05-07 (목) — 1 USD = 1456.80 KRW
2026-05-06 (수) — 1 USD = 1470.50 KRW
2026-05-04 (월) — 1 USD = 1484.80 KRW
2026-04-30 (목) — 1 USD = 1476.10 KRW

※ 가장 최근 영업일: 2026-05-08. 환율 데이터는 1~2일 지연될 수 있습니다.


### currency.py 검증 결과 ✅

| 항목 | 결과 |
|---|---|
| import 동작 | 정상 (셀 3 sys.path 재사용) |
| @tool 변환 | name·description·args 자동 생성 |
| Q3 응답 시간 | 0.55초 (예상 1초 이내 통과) |
| 5영업일 환율 반환 | 5/8(금)·5/7(목)·5/6(수)·5/4(월)·4/30(목) — 5/5 어린이날 + 5/1 근로자의 날 + 주말 휴장 정상 처리 |
| 옵션 A 분리 | 도구는 raw 환율만 반환, query "100달러"에 환산 안 함 |

**알려진 이슈**: 없음. LOOKBACK_DAYS=14가 5/1 근로자의 날 + 어린이날 연휴까지 커버해 5영업일 보장됨.

**다음 도구**: search_web (Tavily, 선택) — 또는 바로 agent.py